In [1]:
!pip install kagglehub[pandas-datasets]

In [68]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("columbine/imdb-dataset-sentiment-analysis-in-csv-format")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-sentiment-analysis-in-csv-format' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-sentiment-analysis-in-csv-format


In [37]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
import re
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers

In [12]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [69]:
train_df = pd.read_csv('/kaggle/input/imdb-dataset-sentiment-analysis-in-csv-format/Test.csv')
test_df = pd.read_csv('/kaggle/input/imdb-dataset-sentiment-analysis-in-csv-format/Test.csv')
valid_df = pd.read_csv('/kaggle/input/imdb-dataset-sentiment-analysis-in-csv-format/Valid.csv')

In [70]:
stopwords_list = stopwords.words('english')
wn = nltk.stem.WordNetLemmatizer()
def preprocessing(text):
  text = text.lower()
  text = re.sub(r'[^a-z\s]','', text)
  tokenized_text = nltk.word_tokenize(text)
  nostop = [word for word in tokenized_text if word not in stopwords_list]
  preprocessed_text = ' '.join(wn.lemmatize(word) for word in nostop)
  return preprocessed_text

In [71]:
train_df['text'] = train_df['text'].apply(preprocessing)
test_df['text'] = test_df['text'].apply(preprocessing)
valid_df['text'] = valid_df['text'].apply(preprocessing)

,text
0,always wrote series complete stinkfest jim bel...
1,st watched dirsteve purcell typical mary kate ...
2,movie poorly written directed fell asleep minu...
3,interesting thing miryang secret sunshine acto...
4,first read berlin meer didnt expect much thoug...
...,...
4995,kind picture john lassiter would making today ...
4996,must see saw whipped press screening hilarious...
4997,nbc ashamed wouldnt allow child see definitely...
4998,movie clumsy mishmash various ghoststory suspe...


In [21]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer1 = Tokenizer(num_words = 10000, oov_token = 'UNK')
tokenizer1.fit_on_texts(train_df['text'])

In [26]:
tokenizer1.word_index

{'UNK': 1,
 'br': 2,
 'movie': 3,
 'film': 4,
 'one': 5,
 'like': 6,
 'time': 7,
 'good': 8,
 'character': 9,
 'even': 10,
 'story': 11,
 'would': 12,
 'get': 13,
 'see': 14,
 'really': 15,
 'make': 16,
 'scene': 17,
 'much': 18,
 'well': 19,
 'bad': 20,
 'great': 21,
 'people': 22,
 'first': 23,
 'dont': 24,
 'also': 25,
 'show': 26,
 'way': 27,
 'made': 28,
 'thing': 29,
 'think': 30,
 'could': 31,
 'life': 32,
 'watch': 33,
 'know': 34,
 'go': 35,
 'two': 36,
 'plot': 37,
 'many': 38,
 'seen': 39,
 'love': 40,
 'actor': 41,
 'year': 42,
 'end': 43,
 'never': 44,
 'look': 45,
 'say': 46,
 'acting': 47,
 'best': 48,
 'little': 49,
 'ever': 50,
 'take': 51,
 'better': 52,
 'man': 53,
 'still': 54,
 'find': 55,
 'something': 56,
 'come': 57,
 'work': 58,
 'want': 59,
 'part': 60,
 'give': 61,
 'director': 62,
 'lot': 63,
 'im': 64,
 'watching': 65,
 'play': 66,
 'guy': 67,
 'back': 68,
 'real': 69,
 'performance': 70,
 'another': 71,
 'didnt': 72,
 'nothing': 73,
 'though': 74,
 'doesnt

In [27]:
training_sequences = tokenizer1.texts_to_sequences(train_df['text'])
testing_sequences = tokenizer1.texts_to_sequences(test_df['text'])
valid_sequences = tokenizer1.texts_to_sequences(valid_df['text'])

In [28]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
train_Sequences_padded = pad_sequences(training_sequences, maxlen=256, padding='post', truncating="post")
test_Sequences_padded = pad_sequences(testing_sequences, maxlen=256, padding='post', truncating="post")
valid_Sequences_padded = pad_sequences(valid_sequences, maxlen=256, padding='post', truncating="post")

In [31]:
len(train_Sequences_padded[2])

256

In [34]:
trainingTensor_df = tf.data.Dataset.from_tensor_slices((train_Sequences_padded, train_df['label'])).shuffle(buffer_size = 1000).batch(64)
testTensor_df = tf.data.Dataset.from_tensor_slices((train_Sequences_padded, test_df['label'])).shuffle(buffer_size = 1000).batch(64)
validationTensor_df = tf.data.Dataset.from_tensor_slices((train_Sequences_padded, valid_df['label'])).shuffle(buffer_size = 1000).batch(64)

In [35]:
from tensorflow.keras.datasets import imdb
(x_train, y_train), (x_test, y_test) = imdb.load_data() # importing through tensorflow

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [39]:
model = Sequential(
    [
      layers.Dense(512, activation='relu'),
      layers.Dense(128, activation='relu'),
      layers.Dense(1, activation='sigmoid')
    ]
)

In [40]:
model.compile(optimizer = 'rmsprop', loss = 'binary_crossentropy', metrics=['accuracy'])

In [42]:
model.fit(trainingTensor_df, epochs = 10, verbose = 1, validation_data=validationTensor_df)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.4976 - loss: 291.5844 - val_accuracy: 0.5108 - val_loss: 3.5759
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5218 - loss: 1.4329 - val_accuracy: 0.4996 - val_loss: 1.0664
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4961 - loss: 0.7833 - val_accuracy: 0.4984 - val_loss: 0.9283
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5247 - loss: 0.7302 - val_accuracy: 0.4984 - val_loss: 0.9752
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5419 - loss: 0.7599 - val_accuracy: 0.4998 - val_loss: 2.1016
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5339 - loss: 0.8518 - val_accuracy: 0.4994 - val_loss: 1.0166
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5281 - loss: 0.7728 - val_accuracy: 0.4960 - val_loss: 1.1665
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5354 - loss: 0.8308 - val_accuracy: 0.4970 - val_lo

In [46]:
model1 = Sequential(
    [
        layers.Embedding(10000, 512),
        layers.GlobalAveragePooling1D(),
        layers.Dense(512, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ]
)

In [50]:
model1.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics=['accuracy'])


In [51]:
model1.fit(trainingTensor_df, epochs = 10, verbose = 1, validation_data=validationTensor_df)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.5562 - loss: 0.6761 - val_accuracy: 0.4972 - val_loss: 0.8221
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7199 - loss: 0.5300 - val_accuracy: 0.4940 - val_loss: 1.1888
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7871 - loss: 0.4318 - val_accuracy: 0.4944 - val_loss: 0.8927
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8530 - loss: 0.3715 - val_accuracy: 0.5020 - val_loss: 1.7818
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8645 - loss: 0.2907 - val_accuracy: 0.5008 - val_loss: 2.0212
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8912 - loss: 0.2533 - val_accuracy: 0.4948 - val_loss: 2.6458
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8196 - loss: 0.4746 - val_accuracy: 0.4984 - val_loss: 2.7046
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9103 - loss: 0.2158 - val_accuracy: 0.5004 - val_los

In [49]:
model1.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 256, 512)       │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 512)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,028,484 (42.07 MB)

 Trainable params: 5,514,241 (21.04 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,514,243 (21.04 MB)

In [53]:
# functional api
input = keras.Input(shape=(None,))
x = layers.Embedding(10000, 512)(input)
x = layers.GlobalAveragePooling1D()(x)
y = layers.Dense(512, activation='relu')(x)
y = layers.Dense(256, activation='relu')(y)
output = layers.Dense(1, activation='sigmoid')(y)


In [55]:
model2 = keras.Model(inputs = input, outputs = output)

In [56]:
model2.compile(optimizer = keras.optimizers.Adam(), loss = keras.losses.BinaryCrossentropy(), metrics=['accuracy'])


In [57]:
model2.fit(trainingTensor_df, epochs = 10, verbose = 1)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.5194 - loss: 0.6910
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6979 - loss: 0.5914
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7000 - loss: 0.6917
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8769 - loss: 0.2981
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8657 - loss: 0.3067
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9101 - loss: 0.2291
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9033 - loss: 0.2624
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9572 - loss: 0.1198
Epoch 9/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9372 - loss: 0.1424
Epoch 10/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9519 - loss: 0.1224


In [63]:
# model subclassing
class TextClassification(keras.Model):
  def __init__(self, input_dim, output_dim, num_classes):
    super(TextClassification, self).__init__()
    self.input_dim = input_dim
    self.output_dim = output_dim
    self.embedding = layers.Embedding(input_dim, output_dim)
    self.avg = layers.GlobalAveragePooling1D()
    self.hidden1 = layers.Dense(512, activation='relu')
    self.hidden2 = layers.Dense(256, activation='relu')
    self.output1 = layers.Dense(num_classes, activation='sigmoid')

  def call(self, input):
    x = self.embedding(input)
    x = self.avg(x)
    h = self.hidden1(x)
    h = self.hidden2(h)
    y = self.output1(h)
    return y

In [64]:
model3 = TextClassification(10000, 512, 1)

In [65]:
model3.compile(optimizer = keras.optimizers.Adam(), loss = keras.losses.BinaryCrossentropy(), metrics=['accuracy'])


In [66]:
model3.fit(trainingTensor_df, epochs = 10, verbose = 1)

Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.5266 - loss: 0.6924
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6661 - loss: 0.6133
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7891 - loss: 0.4751
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8074 - loss: 0.4369
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8817 - loss: 0.2726
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8913 - loss: 0.2737
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9534 - loss: 0.1297
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9292 - loss: 0.2019
Epoch 9/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9446 - loss: 0.1397
Epoch 10/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9656 - loss: 0.1054


In [67]:
model3.summary()

Model: "text_classification_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 256, 512)       │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,542,725 (63.11 MB)

 Trainable params: 5,514,241 (21.04 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 11,028,484 (42.07 MB)